# 2. Premier modèle : TF-IDF + régression logistique

**Le problème** : un modèle ne comprend que des nombres. Il faut transformer chaque avis en une liste de nombres.

**L'idée de TF-IDF** : décrire un avis par **les mots qu'il contient**, en donnant plus de poids aux mots *parlants* (« génial », « nul ») qu'aux mots présents partout (« film »).

Ce notebook a 3 parties :
1. comprendre TF-IDF sur 3 petits avis (je l'ai écrit, tu lis et tu exécutes) ;
2. entraîner le modèle sur les 160 000 avis (**À TOI**) ;
3. regarder ce que le modèle a appris (**À TOI**).

## 2.1 Étape A : compter les mots (« sac de mots »)

On prend 3 mini-avis. `CountVectorizer` fait deux choses :
1. il **découpe** chaque avis en mots (la *tokenisation*), en mettant tout en minuscules et en enlevant la ponctuation ;
2. il **compte** combien de fois chaque mot apparaît dans chaque avis.

Résultat : un tableau avec **une ligne par avis** et **une colonne par mot du vocabulaire**.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

mini_avis = [
    "Film génial, acteurs géniaux !",
    "Film nul. Scénario nul.",
    "Pas mauvais, un film sympa.",
]

compteur = CountVectorizer()
X_compte = compteur.fit_transform(mini_avis)

pd.DataFrame(X_compte.toarray(), columns=compteur.get_feature_names_out())

,acteurs,film,génial,géniaux,mauvais,nul,pas,scénario,sympa,un
0,1,1,1,1,0,0,0,0,0,0
1,0,1,0,0,0,2,0,1,0,0
2,0,1,0,0,1,0,1,0,1,1


**Observe le tableau** :
- l'avis 2 a un **2** dans la colonne `nul` : le mot apparaît deux fois ;
- `génial` et `géniaux` sont deux colonnes **différentes** : pour l'ordinateur, ce sont deux mots sans rapport ;
- **l'ordre des mots a disparu** : « pas mauvais » devient juste « il y a *pas* » et « il y a *mauvais* ». C'est pour ça qu'on appelle cette méthode un **sac de mots** : on vide les mots dans un sac, on perd la phrase.

**Problème** : le mot `film` est dans les 3 avis. Il ne nous aide pas du tout à distinguer un avis positif d'un négatif, mais il compte autant que `génial`.

## 2.2 Étape B : pondérer les mots (TF-IDF)

TF-IDF corrige ce problème en multipliant deux choses :

- **TF** (*term frequency*) : combien de fois le mot est dans **cet** avis. C'est le comptage d'avant.
- **IDF** (*inverse document frequency*) : est-ce que le mot est **rare** dans l'ensemble des avis ? Un mot présent partout (`film`) reçoit un petit poids, un mot rare (`nul`) un grand poids.

**TF-IDF = TF × IDF**, puis chaque ligne est mise à la même échelle (pour qu'un avis long n'ait pas des nombres plus grands qu'un avis court).

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_mini = TfidfVectorizer()
X_tfidf = tfidf_mini.fit_transform(mini_avis)

pd.DataFrame(X_tfidf.toarray(), columns=tfidf_mini.get_feature_names_out()).round(2)

,acteurs,film,génial,géniaux,mauvais,nul,pas,scénario,sympa,un
0,0.55,0.32,0.55,0.55,0.00,0.00,0.00,0.00,0.00,0.00
1,0.00,0.26,0.00,0.00,0.00,0.86,0.00,0.43,0.00,0.00
2,0.00,0.28,0.00,0.00,0.48,0.00,0.48,0.00,0.48,0.48


In [3]:
# Le poids IDF de chaque mot : plus il est grand, plus le mot est rare
pd.Series(tfidf_mini.idf_, index=tfidf_mini.get_feature_names_out()).round(2)

acteurs     1.69
film        1.00
génial      1.69
géniaux     1.69
mauvais     1.69
nul         1.69
pas         1.69
scénario    1.69
sympa       1.69
un          1.69
dtype: float64

**Observe** :
- `film` a l'IDF le plus petit (1.0) car il est dans tous les avis ; dans le tableau, sa valeur est faible (≈ 0.3) ;
- `nul` vaut **0.86** dans l'avis 2 : il est rare **et** répété deux fois, c'est le mot le plus important de l'avis.

**Questions pour toi** (réponds dans la cellule de texte suivante) :
1. Pourquoi `film` a-t-il un poids plus faible que `génial` dans l'avis 1 ?
2. Dans l'avis 3, les mots `pas` et `mauvais` ont le même poids que `sympa`. Pourquoi est-ce un problème pour classer cet avis ?

**Mes réponses** :

1. car film ce repete dans les autres avis alors que genial non 
2. car on sait pas l'ordre "mauvais, pas sympa" ce n'est la meme chose que "pas mauvais, sympa" l'ordre est vraiment important.

## 2.3 À TOI : transformer les 160 000 avis

Maintenant on fait la même chose sur les vrais avis.

**Règle très importante : `fit` uniquement sur le train.**
- `fit_transform(train)` : le vectoriseur **apprend le vocabulaire et les IDF** sur le train, puis transforme le train ;
- `transform(val)` : il transforme la validation **avec le vocabulaire déjà appris**, sans rien réapprendre.

Si on faisait `fit` sur la validation, le modèle « verrait » des informations sur les données qu'on utilise pour le noter. C'est une **fuite de données** (*data leakage*), et la note serait faussement bonne.

**Exercice** :
1. Recharge `train` et `val` (comme au notebook 1, avec `pd.read_parquet`).
2. Crée `vectoriseur = TfidfVectorizer(max_features=50_000)`. (`max_features` garde seulement les 50 000 mots les plus fréquents, pour limiter la mémoire.)
3. Crée `X_train` avec `fit_transform` sur `train["review"]`, et `X_val` avec `transform` sur `val["review"]`.
4. Affiche `X_train.shape` : combien de lignes ? Combien de colonnes ? Est-ce logique ?

In [4]:
# À TOI : recharger les données
train=pd.read_parquet("data/train.parquet")
val=pd.read_parquet("data/validation.parquet")


In [5]:
# À TOI : créer le vectoriseur et transformer train et val
vectoriseur=TfidfVectorizer(max_features=50000)
X_train=vectoriseur.fit_transform(train["review"])
X_val=vectoriseur.transform(val["review"])
print(train.shape)
print(X_train.shape)

(160000, 2)
(160000, 50000)


160000 lignes et 50000 colonnes ce qui est logique car le nombre des avis deja present c'est 160000 ce qui est le nombre de ligne et dans les colonnes il y a 50000 c'est le nombre de mots qui sont presents dans notres avis limites car on a fait max_features=50000.

## 2.4 À TOI : entraîner la régression logistique

C'est le même modèle qu'au Titanic. La seule différence : au lieu de colonnes comme `Age` ou `Sex`, chaque colonne est maintenant un **mot**.

**Exercice** :
1. Crée `modele = LogisticRegression(max_iter=1000)` (importe-le depuis `sklearn.linear_model`).
2. Entraîne-le sur `X_train` et `train["label"]`.
3. Prédis sur `X_val`, puis calcule l'accuracy avec `accuracy_score` (depuis `sklearn.metrics`).

`max_iter=1000` : le modèle s'améliore par étapes successives ; on l'autorise à faire jusqu'à 1000 étapes pour qu'il ait le temps de converger.

**Question** : ton score est-il bien au-dessus du plancher de 50 % qu'on a trouvé au notebook 1 ?

In [6]:
# À TOI : entraîner et évaluer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
modele=LogisticRegression(max_iter=1000)
modele.fit(X_train,train["label"])
y_predict=modele.predict(X_val)
accuracy_score(val["label"],y_predict)


0.9253

**Mon score sur la validation** : 0.9253

---
# Partie 2 : comprendre ce que le modèle a appris

## 2.5 Ouvrir le modèle

Une régression logistique est un modèle **transparent** : elle donne un **coefficient** à chaque colonne, donc ici à chaque mot.
- coefficient **positif** → le mot pousse vers « avis positif » ;
- coefficient **négatif** → le mot pousse vers « avis négatif » ;
- plus le nombre est grand (en valeur absolue), plus le mot pèse.

Pour classer un avis, le modèle additionne (en gros) les coefficients de ses mots, pondérés par leur TF-IDF. Si le total est positif, il prédit « positif ».

`modele.coef_[0]` contient les 50 000 coefficients, dans le même ordre que `vectoriseur.get_feature_names_out()`. On les met ensemble dans une `Series` pour pouvoir les trier.

In [7]:
coefs = pd.Series(modele.coef_[0], index=vectoriseur.get_feature_names_out())

print("Les 15 mots les plus POSITIFS :")
print(coefs.sort_values(ascending=False).head(15).round(2))
print()
print("Les 15 mots les plus NÉGATIFS :")
print(coefs.sort_values().head(15).round(2))

Les 15 mots les plus POSITIFS :
excellent       11.72
magnifique       9.01
superbe          8.17
bon              7.51
adoré            7.28
merveille        7.15
parfaitement     6.83
bijou            6.81
excellente       6.73
parfait          6.49
remarquable      6.21
chef             6.10
sublime          5.99
bémol            5.97
émouvant         5.93
dtype: float64

Les 15 mots les plus NÉGATIFS :
mauvais           -10.38
ennuyeux          -10.08
déception          -9.87
ennui              -9.65
intérêt            -9.56
navet              -9.11
rien               -8.79
décevant           -8.67
plv                -7.86
ridicule           -7.65
nul                -7.48
plat               -7.04
médiocre           -6.92
raté               -6.76
malheureusement    -6.48
dtype: float64


**Questions** (réponds dans la cellule suivante) :
1. Les mots positifs et négatifs te semblent-ils logiques ?
2. Dans les mots négatifs, il y a **au moins 2 mots surprenants** (des mots qui ne sont pas négatifs en français). Lesquels ?
3. **Enquête** : pour chaque mot surprenant, cherche dans `train` les avis qui le contiennent (même méthode qu'au 1.6 avec `.str.contains`). Combien y en a-t-il, quelle proportion est positive (`["label"].mean()`), et lis-en 2 ou 3. Pourquoi le modèle a-t-il appris que ce mot est négatif ?

*Indice : pour chercher un mot entier, et pas un morceau de mot, utilise `.str.contains(r"\bmot\b", case=False)`. `\b` veut dire « début ou fin de mot ».*

In [8]:
# À TOI : enquête sur les mots surprenants
pieges=train["review"].str.contains(r"\bplv\b",case=False)
pieges2=train["review"].str.contains(r"\bintérêt\b",case=False)
#pieges2=train[(train["review"].str.contains("intérêt",case=False))]
#pieges2["review"].tolist()
train.groupby(pieges)["label"].mean()


review
False    0.504457
True     0.015504
Name: label, dtype: float64

In [9]:
train.groupby(pieges2)["label"].mean()
#pieges2["review"].tolist()

review
False    0.522976
True     0.140700
Name: label, dtype: float64

**Mes réponses** :

1. les mots sont logiques sauf 2 dans les negatives
2. intérêt et plv
3. intérêt : 86 % des avis qui le contiennent sont négatifs, parce qu’il est surtout utilisé dans « sans intérêt » ou « aucun intérêt ». Le mot n’est pas négatif en lui-même, c’est son contexte. TF-IDF voit les mots séparément (sac de mots), donc il attribue le côté négatif à « intérêt », et pénalisera à tort « un film d’un grand intérêt ».

plv : 98,5 % des avis qui le contiennent sont négatifs. Il apparaît presque toujours sous la forme « PLV : » à la fin de l’avis : c’est probablement la signature d’un seul auteur, qui écrit surtout des critiques négatives. Le modèle n’a pas appris un mot négatif, il a appris à reconnaître cet auteur. C’est une corrélation trompeuse : elle aide sur ces données mais ne se généraliserait pas à d’autres avis.

## 2.6 À TOI : tester tes avis pièges

Pour un nouvel avis, on suit le même chemin qu'avec la validation : `transform` puis `predict`.

On va utiliser **`predict_proba`** plutôt que `predict` : au lieu de répondre juste 0 ou 1, le modèle donne sa **probabilité** pour chaque classe. Par exemple `[[0.2, 0.8]]` veut dire « 20 % négatif, 80 % positif ». C'est plus instructif : on voit si le modèle hésite ou s'il est sûr de lui.

**Attention** : `transform` attend une **liste** d'avis, même s'il n'y en a qu'un : `vectoriseur.transform([mon_avis])`, avec les crochets.

**Exercice** :
1. Copie tes 2 avis pièges dans deux variables `piege_1` et `piege_2`.
2. Pour chacun, affiche `predict_proba`.
3. Le modèle a-t-il raison ? (Rappel : piège 1 est **négatif**, piège 2 est **positif**.)

In [10]:
# À TOI : tester les avis pièges
pieges_1="Après avoir réussi à mêler film horreur et teen-movie avec le génial Destination finale, le réalisateur James Wong s'est embarqué sous les sirènes d'Hollywood dans un film de science-fiction bourré d'action et d'effets spéciaux. Sur le papelard, The One avait l'air plutôt intéressant avec cette histoire de combat entre personnages de plusieurs dimensions parallèles. Et puis voir Jet Li dans un double rôle était une idée alléchante pour les fans de la star asiatique. En somme, The One promettait un bon film d'action science-fictionnel bourré d'affrontements câblés et d'effets visuels Matrixiens. Verdict : c'est sympa, c'est marrant, c'est bien foutu mais c'est hélas aussi très lourd. Tournant en rond au bout d'une vingtaine de minutes, le long-métrage s'avère très répétitif et surtout très peu captivant au final, le scénario n'arrivant jamais à aller plus loin que le bout de son nez. Quant au casting, c'est également peu convaincant... Jet Li peine à convaincre dans son double rôle de gentil paumé et de méchant diabolique tandis que son comparse de fortune Jason Statham (Ghosts of Mars) se régale à faire le big boy qui flingue tout sur son passage. S'ajoutent également au casting le polyvalent Delroy Lindo et la sexy Carla Gugino, campant respectivement le traqueur du bad guy et la comparse du héros. The One est donc une série B tout ce qu'il y a de plus banale où scènes d'action filmées au ralenti, effets visuels à outrance et musique rock omniprésente se fracassent constamment dans un gros bordel pas toujours digérable. Et si le film confirme une belle place dans le cinéma d'action pour Jason Statham, il nous fait amèrement regretter le passage Outre-Atlantique d'un Jet Li confiné aux rôles indigestes voire nanardesques."
pieges_2= 'Film pas mauvais, qui se laisse regarder. Très belle performance de Wahlberg pas habitué à jouer ce genre de rôle et qui réussit son pari.'
pieges_3="Un film d'un grand intérêt, pas mauvais du tout."
X_pieges_1=vectoriseur.transform([pieges_1])
X_pieges_2=vectoriseur.transform([pieges_2])
X_pieges_3=vectoriseur.transform([pieges_3])


print(modele.predict_proba(X_pieges_1))
print(modele.predict_proba(X_pieges_2))
print(modele.predict_proba(X_pieges_3))



[[0.73403288 0.26596712]]
[[0.70387339 0.29612661]]
[[9.99954895e-01 4.51054096e-05]]


**Résultat** : le modele n'as pas raison 
l'avis num 1: Le modele ne tombe pas dans le piege car il est long et contient plusieurs mots qui confirme que l'avis est negative
l'avis num 2: le modele tombe dans le piege vu que le modele traite mot par mot et donc il tombe sur mauvais qui le considere comme negative et ne comprend pas le contexte que pas mauvais est plutot positive 
l'avis num 3: le modele tombe dans le piege et il est plutot sur avec une proba de 0.99 alors que pas mauvais et un grand interet c'est positive 

## 2.7 À TOI : l'enquête « pas mauvais »

Si le modèle s'est trompé sur un avis, il faut comprendre **pourquoi**. Deux hypothèses :
- **A.** C'est la limite du sac de mots : le modèle voit « pas » et « mauvais » séparément, et les deux sont négatifs.
- **B.** Autre chose…

**Exercice** :
1. Affiche le coefficient de `"pas"` et celui de `"mauvais"` (avec `coefs["pas"]`). Que remarques-tu ?
2. Dans `train`, prends tous les avis qui contiennent `"pas mauvais"`. **Quelle proportion est positive ?**
3. Lis 3 de ces avis. Comment les gens utilisent-ils « pas mauvais » sur Allociné ?
4. **Conclusion** : est-ce que le modèle « se trompe », ou est-ce qu'il a bien appris ce que disent les données ?

In [11]:
# À TOI : enquête "pas mauvais"
print(coefs["pas"])
print(coefs["mauvais"])
pieges=train["review"].str.contains(r"\bpas mauvais\b",case=False)
t_pieges=train[pieges]
print(train.groupby(pieges)["label"].mean())
t_pieges["review"].head(3).to_list()



-6.235197092763113
-10.375441994630949
review
False    0.504897
True     0.116601
Name: label, dtype: float64


["Deux savants sont enlevés par des extraterrestres qui veulent sauver leur propre monde, menacé de destruction par une guerre nucléaire... Une chose est clair, c'est que je ne peux considérer ce film de science-fiction comme étant un des plus réussis de cette époque. Ce qui pose problème ici c'est, d'une part, la mise en scène de Joseph M. Newman qui à tout de même bien mal vieilli et qui propose finalement peu de séquences marquantes et, d'autre part, son casting que l'on sent pas tellement à l'aise - mention spécial à Faith Domergue qui est si peu impliquer que l'ont se demande se qu'elle peut bien faire dans ce film. Mais bon tout n'est pas mauvais non plus, car l'histoire est quand même bien fun à suivre, les effets spéciaux sont plutôt réussi pour l'époque, et en plus la photographie est bien agréable à visionner. L'ensemble se regarde donc facilement jusqu'à la fin, même si je m'attendais au départ à une oeuvre bien plus captivante.",
 'Surfant sur la vague des poupées tueuses, 

**Ma conclusion** : 

.Les coefficients : « pas » (−6,2) et « mauvais » (−10,4) sont tous les deux négatifs. Comme TF-IDF traite les mots séparément (sac de mots), un avis contenant « pas mauvais » additionne deux poids négatifs.

.Ce que disent les données : seulement 12 % des avis d’entraînement qui contiennent « pas mauvais » sont positifs, contre 50 % pour les autres. Donc 88 % sont négatifs.

.Pourquoi : en lisant ces avis, on voit que « pas mauvais » est surtout utilisé comme une concession dans un avis négatif (« le film est raté… mais tout n’est pas mauvais »), et rarement comme un compliment.

.Conclusion : le modèle ne se trompe pas vraiment. Il a appris l’usage réel de « pas mauvais » sur Allociné, pas sa grammaire. Mon avis piège n°2 (« Film pas mauvais, qui se laisse regarder ») est l’exception : c’est pour ça que le modèle se trompe dessus.

.Leçon : un modèle n’apprend pas la langue, il apprend les données. Quand une erreur semble illogique, il faut d’abord vérifier ce que disent les données avant d’accuser la méthode.

## 2.8 À TOI : les bigrammes, une amélioration simple

**Idée** : au lieu de prendre seulement les mots seuls, on prend aussi les **paires de mots qui se suivent**. On appelle ça des **bigrammes** (et les mots seuls des *unigrammes*).

Exemple avec « pas mauvais du tout » :
- unigrammes : `pas`, `mauvais`, `du`, `tout`
- bigrammes : `pas mauvais`, `mauvais du`, `du tout`

Avec les bigrammes, « pas mauvais » devient **une colonne à part**, avec son propre coefficient. On récupère un petit morceau de l'ordre des mots.

Dans scikit-learn, c'est un seul paramètre : `TfidfVectorizer(ngram_range=(1, 2))` veut dire « des 1-grammes aux 2-grammes ». Comme il y a beaucoup plus de paires que de mots, on monte `max_features` à `200_000`.

**Exercice** :
1. Crée un nouveau vectoriseur `vectoriseur_bi` avec `ngram_range=(1, 2)` et `max_features=200_000`, puis refais `X_train_bi` et `X_val_bi` (même règle : `fit` seulement sur le train). *Ça prend environ 1 minute.*
2. Entraîne un nouveau `modele_bi` et calcule son accuracy sur la validation. Est-ce mieux que 92,5 % ?
3. Affiche les 15 « mots » les plus positifs de `modele_bi`. Quels bigrammes apparaissent ? Lequel montre que le modèle a appris une **négation** ?
4. Reteste tes 2 avis pièges avec `modele_bi`. La probabilité a-t-elle bougé ?

In [12]:
# À TOI : vectoriseur avec bigrammes

vectoriseur_bi=TfidfVectorizer(ngram_range=(1,2),max_features=200000)
X_train_bi=vectoriseur_bi.fit_transform(train["review"])
X_val_bi=vectoriseur_bi.transform(val["review"])
print(train.shape)
print(X_train_bi.shape)

(160000, 2)
(160000, 200000)


In [13]:
# À TOI : modèle, accuracy, mots les plus positifs, avis pièges
#modele
modele_bi=LogisticRegression(max_iter=1000)
modele_bi.fit(X_train_bi,train["label"])
y_predict_bi=modele_bi.predict(X_val_bi)

#accuracy
print(accuracy_score(val["label"],y_predict_bi))

#mots les plus positives 
coefs1 = pd.Series(modele_bi.coef_[0],index=vectoriseur_bi.get_feature_names_out())

print("Les 15 mots les plus POSITIFS :")
print(coefs1.sort_values(ascending=False).head(15).round(2))
print()
print("Les 15 mots les plus NÉGATIFS :")
print(coefs1.sort_values().head(15).round(2))


pieges_1="Après avoir réussi à mêler film horreur et teen-movie avec le génial Destination finale, le réalisateur James Wong s'est embarqué sous les sirènes d'Hollywood dans un film de science-fiction bourré d'action et d'effets spéciaux. Sur le papelard, The One avait l'air plutôt intéressant avec cette histoire de combat entre personnages de plusieurs dimensions parallèles. Et puis voir Jet Li dans un double rôle était une idée alléchante pour les fans de la star asiatique. En somme, The One promettait un bon film d'action science-fictionnel bourré d'affrontements câblés et d'effets visuels Matrixiens. Verdict : c'est sympa, c'est marrant, c'est bien foutu mais c'est hélas aussi très lourd. Tournant en rond au bout d'une vingtaine de minutes, le long-métrage s'avère très répétitif et surtout très peu captivant au final, le scénario n'arrivant jamais à aller plus loin que le bout de son nez. Quant au casting, c'est également peu convaincant... Jet Li peine à convaincre dans son double rôle de gentil paumé et de méchant diabolique tandis que son comparse de fortune Jason Statham (Ghosts of Mars) se régale à faire le big boy qui flingue tout sur son passage. S'ajoutent également au casting le polyvalent Delroy Lindo et la sexy Carla Gugino, campant respectivement le traqueur du bad guy et la comparse du héros. The One est donc une série B tout ce qu'il y a de plus banale où scènes d'action filmées au ralenti, effets visuels à outrance et musique rock omniprésente se fracassent constamment dans un gros bordel pas toujours digérable. Et si le film confirme une belle place dans le cinéma d'action pour Jason Statham, il nous fait amèrement regretter le passage Outre-Atlantique d'un Jet Li confiné aux rôles indigestes voire nanardesques."
pieges_2= 'Film pas mauvais, qui se laisse regarder. Très belle performance de Wahlberg pas habitué à jouer ce genre de rôle et qui réussit son pari.'
pieges_3="Un film d'un grand intérêt, pas mauvais du tout."
X_pieges_1=vectoriseur_bi.transform([pieges_1])
X_pieges_2=vectoriseur_bi.transform([pieges_2])
X_pieges_3=vectoriseur_bi.transform([pieges_3])


print(modele_bi.predict_proba(X_pieges_1))
print(modele_bi.predict_proba(X_pieges_2))
print(modele_bi.predict_proba(X_pieges_3))
print(coefs1["pas mauvais"])


0.9383
Les 15 mots les plus POSITIFS :
excellent       12.23
magnifique      10.67
superbe          8.87
bon film         7.85
très bon         7.57
parfaitement     7.08
pas mal          7.02
génial           6.72
voir             6.67
parfait          6.62
beau film        6.61
merveille        6.55
excellente       6.50
bravo            6.25
sublime          6.12
dtype: float64

Les 15 mots les plus NÉGATIFS :
mauvais           -11.49
ennuyeux          -11.12
ennui             -10.60
intérêt            -9.99
navet              -9.82
déception          -9.03
rien               -8.80
ridicule           -8.78
nul                -8.62
décevant           -8.36
mal                -8.01
malheureusement    -7.64
plv                -7.16
raté               -7.07
éviter             -7.07
dtype: float64
[[0.77519882 0.22480118]]
[[0.62074525 0.37925475]]
[[0.99459866 0.00540134]]
-0.4310883074223156


**Mes conclusions sur les bigrammes** : 
.l'accuracy a augmente de 92.5% a 93.83%
.oui le modele a apis qlq biagrammes comme (bon film/tres bon/pas mal) et c'est pas mal qui montre qu'il a appris la negation 
.pour les 3 avis sa probabilite a diminue pour le 2 eme avis 
